# LayOutLMV3 Test env

In [1]:

from doctr.models import detection_predictor

def doctr_run(crop):
    model = detection_predictor('db_resnet50', pretrained=True, assume_straight_pages=False, preserve_aspect_ratio=True)
    out = model([crop])
    return out[0]['words']

/home/bas/Documents/Visual Code Repo's/BelHisFirm-BelHisHAAI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import cv2 as cv
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import base64


model = Qwen2VLForConditionalGeneration.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", torch_dtype="auto", device_map="auto")
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", min_pixels=128 * 28 * 28, max_pixels=1028 * 28 * 28)

def qwen(crop):
    _, buffer = cv.imencode('.png', crop)
    encoded_image = base64.b64encode(buffer).decode('utf-8')
    prompt = (f"Transcribe the word. Return only the word, no other text.")
    message = [
                                {
                                    "role": "user",
                                    "content": [
                                        {"type": "image", "image": f"data:image/png;base64,{encoded_image}"},
                                        {"type": "text", "text": prompt},
                                    ],
                                }
                            ]
                        
    text = processor.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(message)
    inputs = processor(
                                text=[text],
                                images=image_inputs,
                                videos=video_inputs,
                                padding=True,
                                return_tensors="pt",
                            )
    inputs = inputs.to("cuda")

    generated_ids = model.generate(**inputs, max_new_tokens=12)
    generated_ids_trimmed = [
                            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
                            ]
    text = processor.batch_decode(
                                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
                            )
    return text[0]

Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  3.21it/s]


In [5]:
import torch 
import gc

del model
del processor

torch.cuda.empty_cache()
torch.cuda.ipc_collect()
gc.collect()

304

In [ ]:
import json
from collections import defaultdict
import os
import matplotlib.pyplot as plt

pad = 25

with open('test/result.json') as f:
    data = json.load(f)

bboxes_by_image = defaultdict(list)
for ann in data["annotations"]:
    bboxes_by_image[ann["image_id"]].append((ann["bbox"], ann["category_id"]))

file_names = []
for image_names in data['images']:
    file_names.append(image_names['file_name'].split('/')[-1])

counter = 0
for image_id, bboxes in bboxes_by_image.items():

    crops = []

    file_name = file_names[counter]

    image_path = os.path.join("/home/bas/Documents/Visual Code Data/BelHisFirm/project-10-at-2025-08-19-07-41-a527dda8/images", file_name)

    image = cv.imread(image_path)

    for bbox in bboxes:
        x, y, w, h = bbox[0]
        x1, y1 = int(x), int(y)
        x2, y2 = int(x + w), int(y + h)

        crop = image[y1:y2, x1:x2]

        crop_padded = cv.copyMakeBorder(
        crop,
        pad, pad, pad, pad,  # top, bottom, left, right
        cv.BORDER_CONSTANT,
        value=[255, 255, 255]
        )

        crops.append((crop_padded, bbox[0]))
    

    for crop in crops:
        type = 
        boxes = doctr_run(crop[0])
        for box in boxes:  
            x1 = int(box[0][0] * w)
            y1 = int(box[0][1] * h)
            x2 = int(box[2][0] * w)
            y2 = int(box[2][1] * h)

            box_crop = crop[y1:y2, x1:x2]
            
            text = qwen(box_crop)
            

        

    counter += 1

ValueError: too many values to unpack (expected 2)